# ML v1.1 - CatBoost и расширенный baseline

Этот ноутбук продолжает эксперимент `v1.0`. В `v1.0` лучшей baseline-моделью стал `RandomForest`, поэтому здесь мы проверяем более подходящий для табличных anti-fraud данных `CatBoost` и дополнительный tree-baseline `ExtraTrees`. Также добавляем анализ порогов и top-k, потому что в банковской задаче важен не только ответ `0/1`, но и ранжированный список клиентов по риск-скору.


In [3]:
from pathlib import Path
import sys
import subprocess
import importlib.util
import warnings

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped:", exc)

BASE_PATH = Path("/content/drive/MyDrive/ieee-db")
DATA_FILE = BASE_PATH / "client_profile_v1_0_shuffled.csv"

RESULTS_PATH = BASE_PATH / "ml_v1_1_results.csv"
THRESHOLDS_PATH = BASE_PATH / "ml_v1_1_thresholds.csv"
TOPK_PATH = BASE_PATH / "ml_v1_1_topk.csv"
CATBOOST_IMPORTANCE_PATH = BASE_PATH / "ml_v1_1_catboost_feature_importance.csv"

print("Файл данных:", DATA_FILE)
print("RESULTS_PATH:", RESULTS_PATH)


Mounted at /content/drive
Файл данных: /content/drive/MyDrive/ieee-db/client_profile_v1_0_shuffled.csv
RESULTS_PATH: /content/drive/MyDrive/ieee-db/ml_v1_1_results.csv


In [4]:
if importlib.util.find_spec("catboost") is None:
    print("CatBoost не найден. Устанавливаю catboost...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
else:
    print("CatBoost уже установлен.")

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


CatBoost не найден. Устанавливаю catboost...


In [5]:
df = pd.read_csv(DATA_FILE)

print("Размер таблицы:", df.shape)
print("Распределение target:")
print(df["profile_fraud_label"].value_counts(dropna=False))
print("Доля fraud:", df["profile_fraud_label"].mean())

display(df.head())


Размер таблицы: (174280, 95)
Распределение target:
profile_fraud_label
0    167785
1      6495
Name: count, dtype: int64
Доля fraud: 0.03726761533165022


,client_id,tx_count_total,tx_amount_sum,tx_amount_mean,tx_amount_median_proxy,tx_amount_std,tx_count_standart_flow,tx_count_high_risk_flow,tx_sum_stanadart_flow_proxy_proxy,tx_sum_high_risk_flow_proxy,tx_freq_per_day,daily_activity_share,avg_inter_tx_seconds,short_turnover_share,amount_repeat_share,odd_amount_share,weighted_amount_repeat_share,cash_out_ratio_proxy,MCC_risk_share_proxy,high_risk_vs_mean,crypto_pattern_score,low_history_flag,history_support_score,productcd_nunique,addr2_nunique,card4_mode,card6_mode,tx_dt_span_days,top_email_domain,profile_fraud_label,identity_present,num_missing_identity,identity_rows,non_null_id_values,device_type_nunique,id_01_mean,id_01_median,id_01_count,id_02_mean,id_02_median,id_02_count,id_03_mean,id_03_median,id_03_count,id_04_mean,id_04_median,id_04_count,id_05_mean,id_05_median,id_05_count,id_06_mean,id_06_median,id_06_count,id_07_mean,id_07_median,id_07_count,id_08_mean,id_08_median,id_08_count,id_09_mean,id_09_median,id_09_count,id_10_mean,id_10_median,id_10_count,id_11_mean,id_11_median,id_11_count,id_12_mode,id_13_mode,id_14_mode,id_15_mode,id_16_mode,id_17_mode,id_18_mode,id_19_mode,id_20_mode,id_21_mode,id_22_mode,id_23_mode,id_24_mode,id_25_mode,id_26_mode,id_27_mode,id_28_mode,id_29_mode,id_30_mode,id_31_mode,id_32_mode,id_33_mode,id_34_mode,id_35_mode,id_36_mode,id_37_mode,id_38_mode
0,12221_170.0_173,1,47.950,47.950000,47.950,0.000000,1,0,47.95,0.000,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.1,1,1,mastercard,debit,0.000000,aol.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5347__14,1,31.191,31.191000,31.191,0.000000,0,1,0.00,31.191,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,1.0,1.0,0.968935,1,0.1,1,0,mastercard,debit,0.000000,gmail.com,0,1,14,1,24,1.0,-10.0,-10.0,1.0,232053.0,232053.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,-2.0,-2.0,1.0,-100.0,-100.0,1.0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,1.0,-6.0,-6.0,1.0,100.0,100.0,1.0,NotFound,52.0,NaN,Found,Found,166.0,13.0,456.0,256.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T
2,7508_420.0_97,3,117.850,39.283333,36.950,7.133645,3,0,117.85,0.000,0.059385,0.058824,2182363.0,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.3,1,1,visa,debit,50.517662,outlook.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17759_126.0_12,1,103.950,103.950000,103.950,0.000000,1,0,103.95,0.000,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.1,1,1,mastercard,debit,0.000000,gmail.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16092_143.0_74,3,3426.550,1142.183333,1325.880,259.786317,3,0,3426.55,0.000,3.000000,1.000000,2160.0,1.0,0.666667,1.0,0.133333,1.0,0.0,0.0,0.000000,1,0.3,1,1,mastercard,debit,0.050000,gmail.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Политика признаков

По умолчанию используем сопоставимый с `v1.0` набор: убираем `client_id`, прямые утечки и признаки с долей пропусков выше 95%. Это нужно, чтобы честно сравнить модели. Если захотим отдельный эксперимент с максимально полным identity-блоком, можно поставить `USE_SPARSE_IDENTITY = True` и вернуть самые разреженные признаки обратно.


In [6]:
TARGET_COL = "profile_fraud_label"
ID_COLS = ["client_id"]
LEAKY_COLS = [c for c in ["tx_count_fraud", "fraud_rate"] if c in df.columns]
HIGH_MISSING_THRESHOLD = 0.95
USE_SPARSE_IDENTITY = False

if USE_SPARSE_IDENTITY:
    high_missing_cols = []
else:
    high_missing_cols = [
        c for c in df.columns
        if c not in [TARGET_COL] + ID_COLS and df[c].isna().mean() > HIGH_MISSING_THRESHOLD
    ]

feature_drop_cols = ID_COLS + LEAKY_COLS + high_missing_cols

X = df.drop(columns=[c for c in feature_drop_cols + [TARGET_COL] if c in df.columns]).copy()
y = df[TARGET_COL].astype(int).copy()

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Удаляем из признаков:", feature_drop_cols)
print("Осталось признаков:", X.shape[1])
print("Числовых признаков:", len(numeric_cols))
print("Категориальных признаков:", len(categorical_cols))
print("Колонки с >95% missing:", high_missing_cols)
print("Категориальные колонки:", categorical_cols)


Удаляем из признаков: ['client_id', 'id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']
Осталось признаков: 82
Числовых признаков: 66
Категориальных признаков: 16
Колонки с >95% missing: ['id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']
Категориальные колонки: ['card4_mode', 'card6_mode', 'top_email_domain', 'id_12_mode', 'id_15_mode', 'id_16_mode', 'id_28_mode', 'id_29_mode', 'id_30_mode', 'id_31_mode', 'id_33_mode', 'id_34_mode', 'id_35_mode', 'id_36_mode', 'id_37_mode', 'id_38_mode']


In [7]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1764705882,
    random_state=42,
    stratify=y_train_val,
)

print("Train:", X_train.shape, "fraud rate:", y_train.mean())
print("Val:  ", X_val.shape, "fraud rate:", y_val.mean())
print("Test: ", X_test.shape, "fraud rate:", y_test.mean())


Train: (121996, 82) fraud rate: 0.037271713826682845
Val:   (26142, 82) fraud rate: 0.037258052176574095
Test:  (26142, 82) fraud rate: 0.037258052176574095


In [8]:
def calc_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "precision_0_5": precision_score(y_true, y_pred, zero_division=0),
        "recall_0_5": recall_score(y_true, y_pred, zero_division=0),
        "f1_0_5": f1_score(y_true, y_pred, zero_division=0),
        "pred_positive_rate": y_pred.mean(),
    }


def build_threshold_table(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
    rows = []
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "pred_positive_rate": y_pred.mean(),
            "pred_positive_count": int(y_pred.sum()),
        })
    return pd.DataFrame(rows)


def build_topk_table(y_true, y_prob, rates=(0.01, 0.03, 0.05, 0.10)):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    order = np.argsort(-y_prob)
    total_positive = y_true.sum()
    rows = []
    for rate in rates:
        k = max(1, int(np.ceil(len(y_true) * rate)))
        selected = order[:k]
        positives_found = int(y_true[selected].sum())
        rows.append({
            "top_rate": rate,
            "top_k": k,
            "precision_at_k": positives_found / k,
            "recall_at_k": positives_found / total_positive if total_positive else 0.0,
            "fraud_found": positives_found,
            "total_fraud": int(total_positive),
        })
    return pd.DataFrame(rows)


## CatBoost

CatBoost обучаем без ручного one-hot кодирования. Категориальные признаки передаем модели явно, а пропуски в них заменяем строкой `MISSING`. По умолчанию используется CPU. Если в Colab включен GPU, можно поменять `TASK_TYPE` на `GPU`.


In [9]:
TASK_TYPE = "GPU"  # Если в Colab включили GPU, можно поставить "GPU".

cat_feature_indices = [X_train.columns.get_loc(c) for c in categorical_cols]


def prepare_catboost_frame(frame):
    out = frame.copy()
    for c in categorical_cols:
        out[c] = out[c].astype("object").where(out[c].notna(), "MISSING").astype(str)
    return out

X_train_cb = prepare_catboost_frame(X_train)
X_val_cb = prepare_catboost_frame(X_val)
X_test_cb = prepare_catboost_frame(X_test)

train_pool = Pool(X_train_cb, y_train, cat_features=cat_feature_indices)
val_pool = Pool(X_val_cb, y_val, cat_features=cat_feature_indices)
test_pool = Pool(X_test_cb, y_test, cat_features=cat_feature_indices)

catboost_model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.04,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    task_type=TASK_TYPE,
    allow_writing_files=False,
    early_stopping_rounds=80,
    verbose=100,
)

print("Обучаю CatBoost...")
catboost_model.fit(train_pool, eval_set=val_pool, use_best_model=True)


Обучаю CatBoost...


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.8385551	test: 0.8329890	best: 0.8329890 (0)	total: 52.5ms	remaining: 1m 2s
100:	learn: 0.8922428	test: 0.8796597	best: 0.8796597 (100)	total: 5.79s	remaining: 1m 2s
200:	learn: 0.9052619	test: 0.8894459	best: 0.8894459 (200)	total: 9.45s	remaining: 47s
300:	learn: 0.9133067	test: 0.8937235	best: 0.8937289 (297)	total: 13.1s	remaining: 39.1s
400:	learn: 0.9197389	test: 0.8965876	best: 0.8966007 (398)	total: 19.2s	remaining: 38.2s
500:	learn: 0.9251030	test: 0.8980862	best: 0.8980862 (500)	total: 22.8s	remaining: 31.8s
600:	learn: 0.9293792	test: 0.8991490	best: 0.8991490 (600)	total: 26.5s	remaining: 26.4s
700:	learn: 0.9330556	test: 0.9000231	best: 0.9000231 (700)	total: 32.4s	remaining: 23s
800:	learn: 0.9367005	test: 0.9010312	best: 0.9010550 (798)	total: 36.2s	remaining: 18s
900:	learn: 0.9398451	test: 0.9017979	best: 0.9018117 (896)	total: 40.3s	remaining: 13.4s
1000:	learn: 0.9426447	test: 0.9020995	best: 0.9021049 (999)	total: 46.8s	remaining: 9.29s
1100:	learn: 0.945

CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, early_stopping_rounds=80, eval_metric='PRAUC', iterations=1200, learning_rate=0.04, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

## ExtraTrees

ExtraTrees нужен как дополнительный tree-baseline. Для него категориальные признаки кодируются через one-hot, потому что sklearn-деревья не умеют напрямую работать со строковыми категориями.


In [10]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])

preprocess_for_trees = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

extra_trees_model = Pipeline(steps=[
    ("preprocess", preprocess_for_trees),
    ("model", ExtraTreesClassifier(
        n_estimators=350,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

print("Обучаю ExtraTrees...")
extra_trees_model.fit(X_train, y_train)


Обучаю ExtraTrees...


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['tx_count_total',
                                                   'tx_amount_sum',
                                                   'tx_amount_mean',
                                                   'tx_amount_median_proxy',
                                                   'tx_amount_std',
                                                   'tx_count_standart_flow',
                                                   'tx_count_high_risk_flow',
                                                   'tx_sum_stanadart_flow_proxy_proxy',
                                                   'tx_sum_high_risk_flow_proxy',
                                                   'tx_freq_...
                                                                                 min_frequency=20))]),
                                                  ['card4_mode', 'card6_mode',
                                                   'top_email_domain',
                                                   'id_12_mode', 'id_15_mode',
                                                   'id_16_mode', 'id_28_mode',
                                                   'id_29_mode', 'id_30_mode',
                                                   'id_31_mode', 'id_33_mode',
                                                   'id_34_mode', 'id_35_mode',
                                                   'id_36_mode', 'id_37_mode',
                                                   'id_38_mode'])])),
                ('model',
                 ExtraTreesClassifier(class_weight='balanced',
                                      min_samples_leaf=3, n_estimators=350,
                                      n_jobs=-1, random_state=42))])

In [11]:
models = {
    "catboost": catboost_model,
    "extra_trees": extra_trees_model,
}


def predict_proba_model(name, model, X_part):
    if name == "catboost":
        X_part_cb = prepare_catboost_frame(X_part)
        return model.predict_proba(X_part_cb)[:, 1]
    return model.predict_proba(X_part)[:, 1]

results = []
for name, model in models.items():
    for split_name, X_part, y_part in [
        ("val", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        y_prob = predict_proba_model(name, model, X_part)
        row = calc_metrics(y_part, y_prob, threshold=0.5)
        row["model"] = name
        row["split"] = split_name
        results.append(row)

results_df = pd.DataFrame(results)
results_df = results_df[[
    "model", "split", "roc_auc", "pr_auc", "precision_0_5",
    "recall_0_5", "f1_0_5", "pred_positive_rate",
]]

results_df.to_csv(RESULTS_PATH, index=False)
print("Таблица метрик сохранена в:", RESULTS_PATH)
display(results_df.sort_values(["split", "pr_auc"], ascending=[True, False]))

best_model_name = (
    results_df[results_df["split"] == "val"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)
best_model = models[best_model_name]
print("Лучшая модель по val PR-AUC:", best_model_name)


Таблица метрик сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_results.csv


,model,split,roc_auc,pr_auc,precision_0_5,recall_0_5,f1_0_5,pred_positive_rate
1,catboost,test,0.892290,0.445821,0.208309,0.741273,0.325225,0.132584
3,extra_trees,test,0.874013,0.409931,0.290454,0.584189,0.387999,0.074937
0,catboost,val,0.893794,0.442341,0.202886,0.736140,0.318101,0.135185
2,extra_trees,val,0.876722,0.388096,0.273273,0.560575,0.367429,0.076429


Лучшая модель по val PR-AUC: catboost


In [12]:
val_prob = predict_proba_model(best_model_name, best_model, X_val)
test_prob = predict_proba_model(best_model_name, best_model, X_test)

thresholds_df = build_threshold_table(y_val, val_prob)
thresholds_df.to_csv(THRESHOLDS_PATH, index=False)
print("Таблица порогов сохранена в:", THRESHOLDS_PATH)
display(thresholds_df)

topk_df = build_topk_table(y_test, test_prob)
topk_df.to_csv(TOPK_PATH, index=False)
print("Top-k таблица сохранена в:", TOPK_PATH)
display(topk_df)

best_test_pred = (test_prob >= 0.5).astype(int)
print("Classification report для лучшей модели на test при threshold=0.5")
print(classification_report(y_test, best_test_pred, digits=4))


Таблица порогов сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_thresholds.csv


,threshold,precision,recall,f1,pred_positive_rate,pred_positive_count
0,0.10,0.054174,0.966119,0.102595,0.664448,17370
1,0.15,0.070443,0.938398,0.131049,0.496328,12975
2,0.20,0.087709,0.915811,0.160086,0.389029,10170
3,0.25,0.104119,0.890144,0.186432,0.318530,8327
4,0.30,0.123749,0.863450,0.216474,0.259965,6796
5,0.35,0.142957,0.840862,0.244368,0.219149,5729
6,0.40,0.160773,0.802875,0.267900,0.186061,4864
7,0.45,0.180682,0.772074,0.292835,0.159207,4162
8,0.50,0.202886,0.736140,0.318101,0.135185,3534
9,0.55,0.229552,0.700205,0.345754,0.113649,2971


Top-k таблица сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_topk.csv


,top_rate,top_k,precision_at_k,recall_at_k,fraud_found,total_fraud
0,0.01,262,0.740458,0.199179,194,974
1,0.03,785,0.521019,0.419918,409,974
2,0.05,1308,0.399083,0.535934,522,974
3,0.10,2615,0.252390,0.677618,660,974


Classification report для лучшей модели на test при threshold=0.5
              precision    recall  f1-score   support

           0     0.9889    0.8910    0.9374     25168
           1     0.2083    0.7413    0.3252       974

    accuracy                         0.8854     26142
   macro avg     0.5986    0.8161    0.6313     26142
weighted avg     0.9598    0.8854    0.9146     26142



In [13]:
if best_model_name == "catboost":
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": catboost_model.get_feature_importance(),
    }).sort_values("importance", ascending=False)

    importance_df.to_csv(CATBOOST_IMPORTANCE_PATH, index=False)
    print("Важности CatBoost сохранены в:", CATBOOST_IMPORTANCE_PATH)
    display(importance_df.head(40))
else:
    print("Лучшая модель не CatBoost, поэтому CatBoost importance не сохраняю как основной результат.")


Важности CatBoost сохранены в: /content/drive/MyDrive/ieee-db/ml_v1_1_catboost_feature_importance.csv


,feature,importance
1,tx_amount_sum,6.862163
27,top_email_domain,6.786222
14,odd_amount_share,6.305739
3,tx_amount_median_proxy,5.399620
2,tx_amount_mean,5.082524
12,short_turnover_share,3.833662
7,tx_sum_stanadart_flow_proxy_proxy,3.437309
11,avg_inter_tx_seconds,3.145844
9,tx_freq_per_day,3.038600
70,id_20_mode,3.012442


## Как читать результат v1.1

Сравниваем первую очередь по `PR-AUC` на validation и test относительно `RandomForest` из `v1.0`, где `PR-AUC` был примерно 0.412 на validation и 0.421 на test. Если CatBoost или ExtraTrees дают более высокий `PR-AUC` и при этом не разваливаются на test, модель можно считать улучшением. После этого смотрим не только threshold 0.5, но и таблицу порогов: для anti-fraud задачи важно выбрать такой режим, при котором объем клиентов на проверку и качество найденных fraud-клиентов соответствует бизнес-сценарию.
